# 1301. Number of Paths with Max Score

## Topic Alignment
- This problem combines path optimization with counting, relevant for multi-objective optimization in ML (maximize metric + count solutions).
- The pattern appears in reward maximization in RL, optimal policy counting, and analyzing solution space diversity.
- Handling multiple DP states (max score + count) simultaneously is crucial for production metrics tracking.

## Metadata 摘要
- Source: https://leetcode.com/problems/number-of-paths-with-max-score/
- Tags: Dynamic Programming, Array, Matrix
- Difficulty: Hard
- Priority: High

## Problem Statement 原题描述
You are given a square `board` of characters. You can move on the board starting at the bottom right square marked with the character `'S'`.

You need to reach the top left square marked with the character `'E'`. The rest of the squares are labeled either with a numeric character `1, 2, ..., 9` or with an obstacle `'X'`. In one move you can go up, left or up-left (diagonally) only if there is no obstacle there.

Return a list `[score, count]` where:
- `score` is the maximum sum of numeric characters you can collect
- `count` is the number of paths with maximum score

The answer may be too large. Return `[score % (10^9 + 7), count % (10^9 + 7)]`.

If there is no path, return `[0, 0]`.

**Constraints**:
- `2 <= board.length == board[i].length <= 100`

## Progressive Hints
- Hint 1: Work backwards from 'S' (bottom-right) to 'E' (top-left).
- Hint 2: Maintain two DP arrays: max_score and count.
- Hint 3: For each cell, check three predecessors: right, down, diagonal.
- Hint 4: If a predecessor has higher score, update; if equal score, add counts.
- Hint 5: Handle obstacles by skipping them.
- Hint 6: 'E' and 'S' contribute 0 to score.
- Hint 7: Use modulo arithmetic throughout to prevent overflow.
- Hint 8: Initialize 'S' cell with score=0, count=1.

## Solution Overview
Use **2D DP with dual states** (max score + count).

**State Definition**:
- `max_score[i][j]`: maximum score reaching cell (i,j) from 'S'
- `count[i][j]`: number of paths with max_score[i][j]

**Recurrence**:
```python
For cell (i,j):
    candidates = [(i+1,j), (i,j+1), (i+1,j+1)]
    max_prev = max(max_score[candidates])
    
    if max_prev == -infinity:
        max_score[i][j] = -infinity  # No path
        count[i][j] = 0
    else:
        max_score[i][j] = max_prev + value[i][j]
        count[i][j] = sum(count[c] for c in candidates 
                          if max_score[c] == max_prev)
```

## Detailed Explanation

### Problem Mechanics

**Movement**: From any cell, can move:
- Up: (i-1, j)
- Left: (i, j-1)
- Up-left diagonal: (i-1, j-1)

**Reverse thinking**: Go backwards from S to E
- Forward (S to E): up, left, up-left
- Backward (E from S): right, down, down-right

**Why backwards?** Bottom-up DP natural for this direction.

---

### Dual-State DP

**Challenge**: Two objectives simultaneously
1. Maximize score
2. Count paths with max score

**Solution**: Maintain both in parallel
- Update max_score first
- Then update count based on which predecessors contributed

**Key insight**: Paths with sub-optimal scores don't matter

---

### Handling Equal Scores

```python
scores = [right_score, down_score, diag_score]
max_score = max(scores)

# Count how many predecessors achieved max_score
count = 0
if right_score == max_score:
    count += count_right
if down_score == max_score:
    count += count_down
if diag_score == max_score:
    count += count_diag
```

---

### Obstacle Handling

Obstacles ('X'): No path through them
- max_score[i][j] = -infinity
- count[i][j] = 0
- Skip when checking predecessors

---

### Modulo Arithmetic

**Count can be huge**: 10^9 + 7 modulo

```python
MOD = 10**9 + 7
count[i][j] = (count1 + count2 + count3) % MOD
```

**Score**: Also modulo (though usually won't overflow)

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| 2D DP (dual state) | O(n²) | O(n²) | Optimal |
| DFS + memo | O(n²) | O(n²) | Same, more complex |
| Dijkstra variant | O(n² log n) | O(n²) | Overkill for this problem |

In [ ]:
from typing import List

class Solution:
    def pathsWithMaxScore(self, board: List[str]) -> List[int]:
        """
        2D DP tracking max score and count of optimal paths.
        
        Time: O(n²)
        Space: O(n²)
        """
        n = len(board)
        MOD = 10**9 + 7
        
        # DP arrays
        max_score = [[float('-inf')] * n for _ in range(n)]
        count = [[0] * n for _ in range(n)]
        
        # Base case: starting position 'S' at bottom-right
        max_score[n-1][n-1] = 0
        count[n-1][n-1] = 1
        
        # Fill DP table (bottom-up, right to left)
        for i in range(n-1, -1, -1):
            for j in range(n-1, -1, -1):
                if board[i][j] == 'X' or (i == n-1 and j == n-1):
                    continue  # Skip obstacles and starting cell
                
                # Get value of current cell
                val = 0 if board[i][j] == 'E' else int(board[i][j])
                
                # Check three possible predecessors
                candidates = []
                if i + 1 < n:  # From below
                    candidates.append((i+1, j))
                if j + 1 < n:  # From right
                    candidates.append((i, j+1))
                if i + 1 < n and j + 1 < n:  # From diagonal
                    candidates.append((i+1, j+1))
                
                if not candidates:
                    continue
                
                # Find maximum score from predecessors
                max_prev = max(max_score[pi][pj] for pi, pj in candidates)
                
                if max_prev == float('-inf'):
                    continue  # No valid path
                
                # Update current cell
                max_score[i][j] = max_prev + val
                
                # Count paths: sum counts from all predecessors with max score
                for pi, pj in candidates:
                    if max_score[pi][pj] == max_prev:
                        count[i][j] = (count[i][j] + count[pi][pj]) % MOD
        
        # Result at top-left 'E'
        if max_score[0][0] == float('-inf'):
            return [0, 0]
        
        return [int(max_score[0][0]) % MOD, count[0][0]]

In [ ]:
# Test cases
tests = [
    (["E23","2X2","12S"], [7, 1]),
    (["E12","1X1","21S"], [4, 2]),
    (["E11","XXX","11S"], [0, 0]),
]

solver = Solution()
for board, expected in tests:
    result = solver.pathsWithMaxScore(board)
    assert result == expected, f"Failed for {board}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n²) where n = board dimension
  - Process each cell once
  - Check 3 predecessors per cell: O(1)
- **Space**: O(n²) for two DP arrays
  - Can optimize to O(n) using rolling arrays

## Edge Cases & Pitfalls
- **No valid path**: Return [0, 0]
- **Single path**: Count = 1
- **All obstacles**: No path
- **Large counts**: Use modulo throughout
- **Common mistake**: Forgetting to check all three directions
- **Common mistake**: Not handling 'E' and 'S' specially (value = 0)
- **Boundary checking**: Ensure indices in bounds

## Follow-up Variants
- **Minimize score**: Find minimum instead of maximum
- **Weighted moves**: Different costs for different directions
- **k-best paths**: Count top-k scoring paths
- **3D board**: Extend to 3D grid

## Takeaways
- **Dual-state DP** handles multiple objectives simultaneously.
- **Reverse direction** sometimes simplifies DP dependencies.
- **Modulo arithmetic** prevents overflow in counting problems.
- Combining maximization + counting is a common pattern.
- This extends to reward optimization with solution diversity metrics.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 62 | Unique Paths | Path counting |
| LC 64 | Minimum Path Sum | Path optimization |
| LC 174 | Dungeon Game | Reverse DP |
| LC 741 | Cherry Pickup | Dual-direction DP |
| LC 931 | Minimum Falling Path Sum | Grid DP |